# 05-02. Geography of PSLNMD  
# PSLNMD を世界地図として理解する

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## 今日の目的 / Goals

05-01 では、4-box から Toggweiler 型 6-box へ進む理由を学びました。  
In 05-01, we learned why we move from the 4-box model to the Toggweiler-style six-box model.

6-box の箱は、

```text
P S L N M D
```

でした。

この Notebook では、それぞれの box を **地理と深さ** のイメージで理解します。  
In this notebook, we understand each box in terms of **geography and depth**.

今日の主題はこれです。  
The main theme is:

> **PSLNMD は単なる記号ではなく、世界海洋を粗く切り分けた地理的・鉛直的な構造である。**  
> **PSLNMD is not just notation; it is a coarse geographic and vertical partition of the world ocean.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: PSLNMD はどこにあるのか / Where are PSLNMD boxes?

6-box モデルでは、表層 box が 4 つあります。  
In the six-box model, there are four surface boxes.

```text
P: Polar surface
S: Subpolar surface
L: Low-latitude surface
N: North Atlantic surface
```

内部 box は 2 つです。  
There are two interior boxes.

```text
M: mid-depth ocean
D: deep ocean
```

ここで重要なのは、P, S, L, N が単に横に並んだ箱ではなく、海洋循環における役割が違うことです。  
The important point is that P, S, L, and N are not just boxes placed side by side; they have different roles in ocean circulation.

## 2. Prediction: 各 box の役割を予想する / Predict the role of each box

コードを書く前に、各 box の役割を予想してみます。  
Before writing code, we predict the role of each box.

### P: Polar surface / 極域表層

```text
広い表層
大気 CO2 と交換する
深層水が戻ってくる経路の一部
```

```text
large surface reservoir
exchanges CO2 with the atmosphere
part of the return pathway from deep water
```

### S: Subpolar surface / 南大洋表層

```text
深層水が表層へ戻る場所
大気と深層をつなぐ窓
CO2 放出・吸収の重要域
```

```text
where deep water returns to the surface
a window connecting atmosphere and deep ocean
important region for CO2 outgassing and uptake
```

### L: Low-latitude surface / 低緯度表層

```text
暖かい表層
生物ポンプが働く
表層経路の一部
```

```text
warm surface ocean
biological pump operates
part of surface pathway
```

### N: North Atlantic surface / 北大西洋表層

```text
沈み込みの入口
内部水形成
大気と内部をつなぐ別の窓
```

```text
entrance of sinking
formation of interior water
another window connecting atmosphere and interior
```

### M: Mid-depth / 中層

```text
比較的新しい内部水
表層と深層の間
D より表層に近い
```

```text
relatively young interior water
between surface and deep ocean
closer to surface than D
```

### D: Deep / 深層

```text
古い水
炭素貯蔵
表層から隔離されやすい
```

```text
old water
carbon storage
more isolated from the surface
```

## 3. 表層 box の地理的イメージ / Geographic image of surface boxes

ここでは、簡単な模式的世界地図を作ります。  
Here we create a simple schematic world map.

正確な海岸線は描きません。  
We do not draw accurate coastlines.

目的は、P, S, L, N が世界海洋のどこを代表しているかを直感的に理解することです。  
The purpose is to intuitively understand which parts of the world ocean P, S, L, and N represent.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))

# draw schematic ocean domain
ax.set_xlim(-180, 180)
ax.set_ylim(-80, 80)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Schematic geography of surface boxes: P, S, L, N")

# box regions
regions = {
    "S": {"lon": (-180, 180), "lat": (-70, -45), "label": "S\nSouthern Ocean"},
    "P": {"lon": (120, 260), "lat": (-35, 35), "label": "P\nPolar surface"},
    "L": {"lon": (-60, 120), "lat": (-35, 35), "label": "L\nLow-latitude surface"},
    "N": {"lon": (-70, 20), "lat": (35, 70), "label": "N\nNorth Atlantic"},
}

for key, r in regions.items():
    lon0, lon1 = r["lon"]
    lat0, lat1 = r["lat"]
    if lon1 > 180:
        # split Pacific across dateline
        ax.add_patch(plt.Rectangle((lon0, lat0), 180-lon0, lat1-lat0, fill=False, linewidth=2))
        ax.add_patch(plt.Rectangle((-180, lat0), lon1-180, lat1-lat0, fill=False, linewidth=2))
        ax.text(160, 0, r["label"], ha="center", va="center")
        ax.text(-150, 0, "P", ha="center", va="center")
    else:
        ax.add_patch(plt.Rectangle((lon0, lat0), lon1-lon0, lat1-lat0, fill=False, linewidth=2))
        ax.text((lon0+lon1)/2, (lat0+lat1)/2, r["label"], ha="center", va="center")

ax.axhline(0, linestyle="--", linewidth=0.8)
ax.grid(True, alpha=0.3)
plt.show()

### Mini exercise 1 / ミニ演習 1

この模式図で、S はなぜ東西に広く描かれているのでしょうか。  
Why is S drawn as a zonally wide region in this schematic?

P と L を分けることには、どのような意味があるでしょうか。  
What is the meaning of separating P and L?

## 4. 鉛直構造のイメージ / Vertical structure

次に、鉛直構造を見ます。  
Next, we look at the vertical structure.

表層は P, S, L, N です。  
The surface boxes are P, S, L, and N.

内部は M と D です。  
The interior boxes are M and D.

```text
surface: P, S, L, N
mid-depth: M
deep: D
```

この鉛直分割により、中層と深層の違いを扱えます。  
This vertical division allows us to represent differences between mid-depth and deep water.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# x positions for surface boxes
x = {"P": 0.15, "S": 0.38, "L": 0.61, "N": 0.84}
y_surface = 0.78

for b in ["P", "S", "L", "N"]:
    ax.text(x[b], y_surface, b, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

ax.text(0.50, 0.48, "M\nMid-depth", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black"))
ax.text(0.50, 0.20, "D\nDeep", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black"))

# conceptual arrows
for b in ["P", "S", "L", "N"]:
    ax.annotate("", xy=(0.50, 0.52), xytext=(x[b], y_surface-0.05),
                arrowprops=dict(arrowstyle="->", alpha=0.5))

ax.annotate("", xy=(0.50, 0.25), xytext=(0.50, 0.43),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(0.38, y_surface-0.05), xytext=(0.50, 0.25),
            arrowprops=dict(arrowstyle="->", linestyle="--"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Vertical structure of PSLNMD")
plt.show()

## 5. Discussion: なぜ M と D を分けるのか / Why separate M and D?

M と D を分ける理由は、単に深さを増やしたいからではありません。  
The reason for separating M and D is not simply to add vertical resolution.

重要なのは、**表層とのつながり方が違う**ことです。  
The important point is that they have different connections to the surface.

### M

M は中層であり、表層と比較的つながりやすい内部水として考えます。  
M is mid-depth water and can be thought of as interior water relatively connected to the surface.

### D

D は深層であり、より古く、炭素を長く貯蔵しやすい箱として考えます。  
D is deep water and can be thought of as older water that stores carbon for longer.

したがって、M と D を分けると、

```text
ventilation age
O2
DIC
Δ14C
```

の違いを考えられるようになります。  
Therefore, separating M and D allows us to consider differences in:

```text
ventilation age
O2
DIC
Δ14C
```

## 6. Box table を作る / Create a box table

Notebook でモデルを扱うときは、box の一覧を DataFrame にしておくと便利です。  
When working with a model in a notebook, it is useful to store box information in a DataFrame.

ここでは、各 box の地理、層、主な役割をまとめます。  
Here we summarize geography, layer, and main role for each box.

In [ ]:
boxes = ["P", "S", "L", "N", "M", "D"]

box_table = pd.DataFrame({
    "Box": boxes,
    "Japanese": ["極域表層", "南大洋表層", "低緯度表層", "北大西洋表層", "中層", "深層"],
    "English": ["Polar surface", "Subpolar surface", "Low-latitude surface",
                "North Atlantic surface", "Mid-depth", "Deep ocean"],
    "Layer": ["surface", "surface", "surface", "surface", "mid-depth", "deep"],
    "Role": [
        "large surface reservoir",
        "upwelling and air-sea exchange",
        "warm surface and biology",
        "sinking region",
        "intermediate pathway",
        "old carbon storage",
    ],
})
box_table

### Mini exercise 2 / ミニ演習 2

`Role` の列を日本語に書き換えてみましょう。  
Rewrite the `Role` column in Japanese.

また、自分なら各 box にどのような説明を加えるか考えてください。  
Also think about what explanation you would add for each box.

In [ ]:
box_table_jp = box_table.copy()
box_table_jp["Role_Japanese"] = [
    "広い表層リザーバー",
    "湧昇と大気海洋交換",
    "暖かい表層と生物生産",
    "沈み込み域",
    "中層の輸送経路",
    "古い炭素の貯蔵庫",
]
box_table_jp

## 7. 仮の体積と深さを設定する / Assign provisional volumes and depths

次に、各 box に仮の体積と代表深度を与えます。  
Next, we assign provisional volumes and representative depths to each box.

これは厳密な値ではありません。  
These are not exact values.

目的は、

```text
表層 box は小さい
内部 box は大きい
D は M より深い
```

という直感を数値として確認することです。  
The goal is to numerically confirm the intuition:

```text
surface boxes are smaller
interior boxes are larger
D is deeper than M
```

In [ ]:
VOCN_TOTAL = 1.292e18

VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())

DEPTH = {
    "P": 100,
    "S": 100,
    "L": 100,
    "N": 100,
    "M": 1000,
    "D": 3500,
}

geom = pd.DataFrame({
    "Box": boxes,
    "Volume [m3]": [VOLUME[b] for b in boxes],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
    "Representative depth [m]": [DEPTH[b] for b in boxes],
})
geom

In [ ]:
plt.figure()
plt.bar(geom["Box"], geom["Volume fraction"])
plt.xlabel("Box")
plt.ylabel("Volume fraction")
plt.title("Provisional box volume fractions")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

plt.figure()
plt.bar(geom["Box"], geom["Representative depth [m]"])
plt.xlabel("Box")
plt.ylabel("Representative depth [m]")
plt.title("Representative depth of PSLNMD boxes")
plt.gca().invert_yaxis()
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## 8. 地理と輸送をつなげる / Connect geography and transport

PSLNMD の地理を理解したら、次に考えるのは輸送です。  
After understanding PSLNMD geography, the next step is transport.

ここでは、最小限の経路を置きます。  
Here we place a minimal pathway:

```text
N → M → D → S → P → L → N
```

これは、次のような概念を表します。  
This represents:

```text
North Atlantic sinking
interior pathway through M and D
Southern Ocean return/upwelling
surface return through P and L
```

```text
北大西洋沈み込み
M と D を通る内部経路
南大洋での戻り・湧昇
P と L を通る表層戻り経路
```

In [ ]:
pathway = [("N", "M"), ("M", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
Q = 2.0e7

pd.DataFrame(pathway, columns=["From", "To"])

## 9. 経路を図に描く / Draw the pathway

地理的な模式図の上に、簡単な輸送経路を描いてみます。  
We now draw the simple transport pathway on a schematic diagram.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pos = {
    "P": (0.12, 0.70),
    "S": (0.36, 0.70),
    "L": (0.60, 0.70),
    "N": (0.84, 0.70),
    "M": (0.50, 0.43),
    "D": (0.50, 0.18),
}

for b, (x, y) in pos.items():
    ax.text(x, y, b, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

for a, b in pathway:
    ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 0.95)
ax.axis("off")
ax.set_title("Conceptual transport pathway in PSLNMD")
plt.show()

### Mini exercise 3 / ミニ演習 3

この経路では、N で沈み込んだ水はどの box を通って S に戻りますか。  
In this pathway, through which boxes does water sinking in N return to S?

また、S から N へ戻る表層経路はどれですか。  
What is the surface pathway from S back to N?

## 10. 水塊年齢を予想する / Predict water-mass age

輸送経路を見たら、次に水塊年齢を予想します。  
After looking at the transport pathway, we predict water-mass age.

一般に、

```text
表層 box: 若い
M: 中間的
D: 古い
```

となるはずです。  
In general, we expect:

```text
surface boxes: young
M: intermediate
D: old
```

特に D は、表層から最も離れており、炭素を長く貯蔵しやすい box です。  
D is especially far from the surface and tends to store carbon for a long time.

In [ ]:
# A very simple ideal-age calculation for intuition
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400
DT_YEAR = DT / SEC_PER_YEAR
volumes = np.array([VOLUME[b] for b in boxes])

def build_flux_matrix(pathway, Q, boxes):
    idx = {b: i for i, b in enumerate(boxes)}
    F = np.zeros((len(boxes), len(boxes)))
    for source, target in pathway:
        i = idx[target]
        j = idx[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

F = build_flux_matrix(pathway, Q, boxes)

def one_step_transport(c):
    return c + (F @ c) * DT / volumes

def one_step_age(age):
    new = one_step_transport(age)
    # age increases in M and D
    for b in ["M", "D"]:
        new[boxes.index(b)] += DT_YEAR
    # surface reset
    for b in ["P", "S", "L", "N"]:
        new[boxes.index(b)] = 0.0
    return new

def run_age(years=3000):
    age = np.zeros(len(boxes))
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for i, b in enumerate(boxes):
                row[b] = age[i]
            hist.append(row)
        age = one_step_age(age)
    return pd.DataFrame(hist)

age = run_age()

plt.figure()
for b in boxes:
    plt.plot(age["year"], age[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("Predicted ideal age in PSLNMD")
plt.legend()
plt.grid(True)
plt.show()

age.tail()

### Mini exercise 4 / ミニ演習 4

M と D のどちらが古くなりましたか。  
Which becomes older, M or D?

この結果は、地理・鉛直構造の直感と合っていますか。  
Is this consistent with the geographic and vertical intuition?

## 11. 05-02 のまとめ / Summary of 05-02

この Notebook で学んだことは次です。  
What we learned:

1. PSLN はすべて表層 box である。  
   P, S, L, and N are all surface boxes.

2. P は極域表層、S は南大洋表層、L は低緯度表層、N は北大西洋表層を表す。  
   P is Polar surface, S is Subpolar surface, L is low-latitude surface, and N is North Atlantic surface.

3. M は中層、D は深層を表す。  
   M is mid-depth water and D is deep water.

4. PSLNMD は、地理的な表層分割と鉛直的な内部構造を組み合わせたモデルである。  
   PSLNMD combines geographic surface partitioning with vertical interior structure.

5. N で沈み込んだ水は M, D を通り、S で表層へ戻るという経路を考えられる。  
   Water sinking in N can be thought of as passing through M and D and returning to the surface in S.

## Key message

> **PSLNMD は、世界海洋を「表層の地理」と「内部の深さ」に分けるための最小構造である。**  
> **PSLNMD is a minimal structure for dividing the world ocean by surface geography and interior depth.**

## 12. 課題 / Exercises

### 課題 1 / Exercise 1

PSLNMD の各 box を、地理と深さの観点から説明せよ。  
Explain each PSLNMD box in terms of geography and depth.

### 課題 2 / Exercise 2

S box が炭素循環において重要である理由を説明せよ。  
Explain why the S box is important for carbon cycling.

### 課題 3 / Exercise 3

P と L を分ける意味を説明せよ。  
Explain the meaning of separating P and L.

### 課題 4 / Exercise 4

M と D を分ける意味を説明せよ。  
Explain the meaning of separating M and D.

### 課題 5 / Exercise 5

N → M → D → S → P → L → N という経路を、水塊の沈み込み・湧昇・表層戻り経路として説明せよ。  
Explain the pathway N → M → D → S → P → L → N in terms of sinking, upwelling, and surface return flow.

## 想定解答 / Expected answers

### 課題 1

P は極域表層、S は南大洋表層、L は低緯度表層、N は北大西洋表層を表す。これらは表層 box である。M は中層、D は深層であり、内部海洋の鉛直構造を表す。  
P is Polar surface, S is Subpolar surface, L is low-latitude surface, and N is North Atlantic surface. These are surface boxes. M is mid-depth and D is deep, representing vertical structure in the ocean interior.

### 課題 2

S は深層水が表層へ戻る場所として重要であり、深層に蓄積された DIC が大気と再びつながる窓になる。そのため、大気 CO2 を考える上で重要である。  
S is important because it is where deep water returns to the surface and where DIC stored in the deep ocean can reconnect with the atmosphere. Therefore, it is important for atmospheric CO2.

### 課題 3

P と L を分けることで、極域表層とその他の低緯度表層を区別できる。これにより、表層海洋の地理的違い、大気 CO2 交換、生物ポンプ、水塊経路の違いを考えられる。  
Separating P and L distinguishes Polar surface from other low-latitude surface waters. This allows geographic differences in surface ocean, air-sea CO2 exchange, biological pump, and water-mass pathways to be considered.

### 課題 4

M と D を分けることで、中層水と深層水の違いを表現できる。これにより、ベンチレーション年齢、O2、DIC、\( \Delta^{14}\mathrm{C} \) などの鉛直差を扱える。  
Separating M and D represents differences between mid-depth and deep water. This allows vertical differences in ventilation age, O2, DIC, and \( \Delta^{14}\mathrm{C} \) to be treated.

### 課題 5

N で沈み込んだ水は M を通って D に入り、深層に炭素を蓄積する。その後、D の水は S で表層へ戻り、P と L を通って N へ戻る。この経路は、内部循環と表層戻り経路を単純化したものである。  
Water sinking in N enters M and then D, storing carbon in the deep ocean. It then returns to the surface in S and moves through P and L back to N. This pathway is a simplified representation of interior circulation and surface return flow.

## 13. 次へ / Next step

次の Notebook では、PSLNMD の輸送行列を本格的に作ります。  
In the next notebook, we build the PSLNMD transport matrix more carefully.

```text
05-03 Transport Matrix of PSLNMD
```

そこで扱う内容は次です。  
We will treat:

```text
輸送行列
質量保存
保存トレーサー
流量を変える実験
デバッグ方法
```

```text
transport matrix
mass conservation
conserved tracers
transport sensitivity experiments
debugging methods
```